In [ ]:
!pip install -q pypdf
!pip install -q langchain-text-splitters
!pip install -q langchain langchain-community langchain-groq \
    faiss-cpu sentence-transformers \
    streamlit python-dotenv tiktoken groq

In [ ]:
import os
from google.colab import userdata

# Set your Groq API key securely
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Quick check
import groq
client = groq.Groq()
print("Groq connected ✓")

In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Upload your PDF
print("Upload your PDF file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {filename}")

# Load pages
loader = PyPDFLoader(filename)
documents = loader.load()
print(f"✓ Loaded {len(documents)} pages")

# Chunk
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"✓ Created {len(chunks)} chunks")

print(f"\n--- Sample Chunk ---")
print(chunks[0].page_content)
print(f"\nMetadata: {chunks[0].metadata}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import time

# Load HuggingFace embedding model (free, no API key needed)
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("✓ Embedding model loaded")

# Build FAISS index from chunks
print(f"\nEmbedding {len(chunks)} chunks into FAISS index...")
start = time.time()
vectorstore = FAISS.from_documents(chunks, embeddings)
elapsed = time.time() - start
print(f"✓ FAISS index built in {elapsed:.1f}s")

# Save index to disk
vectorstore.save_local("faiss_index")
print("✓ Index saved to faiss_index/")

# Quick retrieval test
print("\n--- Retrieval Test ---")
query = "What are the main topics covered in this document?"
results = vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(results):
    print(f"\nResult {i+1} (page {doc.metadata.get('page', '?')}):")
    print(doc.page_content[:200])

In [ ]:
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
import os

# Load Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.environ["GROQ_API_KEY"]
)
print("✓ Groq LLM loaded")

# Load FAISS index
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
print("✓ Retriever ready")

# Prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions based strictly on the provided context.
Always cite the source by mentioning page numbers when available.
If the answer is not in the context, say 'I could not find this in the document.'

Context:
{context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(
        f"[Page {doc.metadata.get('page', '?')}]: {doc.page_content}"
        for doc in docs
    )

# Conversation memory
chat_history = []

def ask(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    return answer, docs

print("✓ RAG chain ready\n")

# Test
answer, sources = ask("What is this document about?")
print("Answer:\n", answer)
print("\nSources:")
for doc in sources:
    print(f"  → Page {doc.metadata.get('page', '?')}: {doc.page_content[:100]}...")

In [ ]:
print("RAG Q&A — type 'exit' to stop\n")
while True:
    question = input("You: ").strip()
    if question.lower() in ["exit", "quit", "q"]:
        print("Exiting.")
        break
    if not question:
        continue
    answer, sources = ask(question)
    print(f"\nAssistant: {answer}")
    pages = [str(doc.metadata.get('page', '?')) for doc in sources]
    print(f"Sources: pages {', '.join(pages)}\n")

In [ ]:
print("RAG Q&A — type 'exit' to stop\n")
while True:
    question = input("You: ").strip()
    if question.lower() in ["exit", "quit", "q"]:
        print("Exiting.")
        break
    if not question:
        continue
    answer, sources = ask(question)
    print(f"\nAssistant: {answer}")
    pages = [str(doc.metadata.get('page', '?')) for doc in sources]
    print(f"Sources: pages {', '.join(pages)}\n")